# Speech Emotion Recognition with Audio Flamingo 3 (NVIDIA) — RAVDESS
Laedt das RAVDESS-Dataset direkt von Kaggle herunter und klassifiziert die Emotion zero-shot mit `nvidia/audio-flamingo-3-hf`.

**Lizenz-Hinweis**: Audio Flamingo 3 steht unter der NVIDIA OneWay Noncommercial License (nur fuer nicht-kommerzielle Forschung). Vergleiche https://huggingface.co/nvidia/audio-flamingo-3.

**Hardware**: 7B Parameter, A100/H100 empfohlen. RAVDESS hat 1{,}440 WAVs, jeweils ~3-4 Sekunden — passt unter dem 10-Minuten-Limit von AF3.

In [1]:
# Audio Flamingo 3 ist in `transformers` integriert, braucht aber eine aktuelle Version (git main).
!pip install -q kaggle librosa
!pip install -q --upgrade "git+https://github.com/huggingface/transformers" accelerate


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install --user "numpy<2"


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os

# Kaggle-Zugangsdaten hier eintragen
os.environ["KAGGLE_USERNAME"] = "DEIN_USERNAME"
os.environ["KAGGLE_KEY"]      = "DEIN_API_KEY"

# RAVDESS Speech-Audio auf Kaggle
DATASET_SLUG = "uwrfkaggler/ravdess-emotional-speech-audio"
DOWNLOAD_DIR = "/tmp/ravdess"

# Prompt-Modus: 'normal' oder 'prosodic'
PROMPT_MODE = "normal"

# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
License(s): CC-BY-NC-SA-4.0
100%|████████████████████████████████████████| 429M/429M [00:10<00:00, 44.2MB/s]



In [4]:
from pathlib import Path
from collections import Counter

# RAVDESS Filename-Encoding (7 Felder, bindestrichgetrennt):
#   03-01-06-01-02-01-12.wav
#   Feld 3 = Emotion: 01 neutral, 02 calm, 03 happy, 04 sad,
#                     05 angry, 06 fearful, 07 disgust, 08 surprised
RAVDESS_EMOTION = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

def ravdess_emotion(path):
    parts = path.stem.split("-")
    return RAVDESS_EMOTION.get(parts[2]) if len(parts) >= 3 else None

# Deduplicate by filename — the Kaggle zip sometimes extracts files at
# multiple directory depths, producing duplicate basenames via rglob.
seen = set()
all_files = []
for f in sorted(Path(DOWNLOAD_DIR).rglob("*.wav")):
    if f.name not in seen:
        seen.add(f.name)
        all_files.append(f)

print(f"Gefundene WAV-Dateien (unique): {len(all_files)}")
for f in all_files[:5]:
    print(f"  {f}  [{ravdess_emotion(f)}]")

gt_counts = Counter(ravdess_emotion(f) for f in all_files)
print(f"\nGround-truth Verteilung: {gt_counts.most_common()}")

Gefundene WAV-Dateien (unique): 1440
  /tmp/ravdess/Actor_01/03-01-01-01-01-01-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-01-02-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-02-01-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-02-02-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-02-01-01-01-01.wav  [calm]

Ground-truth Verteilung: [('calm', 192), ('happy', 192), ('sad', 192), ('angry', 192), ('fearful', 192), ('disgust', 192), ('surprised', 192), ('neutral', 96)]


In [5]:
import json
import torch
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

# Prompt-Set: parallel zur EMO-DB-Version, mit RAVDESS-Labelset.
PROMPTS = {
    "normal": (
        "Listen to this audio clip and classify the emotion of the speaker. "
        "Choose from: neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
        "Reply with just the emotion label."
    ),
    "prosodic": (
        "Listen to this audio clip and classify the emotion of the speaker "
        "based only on prosodic and acoustic features such as pitch, "
        "speech rate, rhythm and loudness. "
        "Ignore the spoken words and linguistic content entirely. "
        "Choose from: neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
        "Reply with just the emotion label."
    ),
}

PROMPT = PROMPTS[PROMPT_MODE]
output_file = f"emotion_results_audioflamingo_ravdess_{PROMPT_MODE}.json"

# Resume
results = {}
if os.path.exists(output_file):
    with open(output_file) as f:
        results = json.load(f)
    print(f"{len(results)} Dateien bereits verarbeitet")

# Modell laden
print("Lade Modell...")
MODEL_ID = "nvidia/audio-flamingo-3-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
).eval()
print(f"Modell geladen auf: {next(model.parameters()).device}")

878 Dateien bereits verarbeitet
Lade Modell...


Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

Modell geladen auf: cuda:0


In [6]:
total_files = len(all_files)

# Audio Flamingo 3 akzeptiert Dateipfade direkt — kein vorheriges librosa-Loading noetig.
for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Uebersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        conversation = [{
            "role": "user",
            "content": [
                {"type": "text",  "text": PROMPT},
                {"type": "audio", "path": str(audio_file)},
            ],
        }]

        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        )
        # Float-Tensoren (Audio-Features) muessen in das Modell-dtype (bf16)
        # gecastet werden; int-Tensoren (input_ids, attention_mask) bleiben.
        inputs = {
            k: (v.to(model.device, dtype=torch.bfloat16) if torch.is_tensor(v) and v.is_floating_point()
                else v.to(model.device) if torch.is_tensor(v)
                else v)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            generate_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateien verarbeitet.")

(1/1440) Uebersprungen: 03-01-01-01-01-01-01.wav
(2/1440) Uebersprungen: 03-01-01-01-01-02-01.wav
(3/1440) Uebersprungen: 03-01-01-01-02-01-01.wav
(4/1440) Uebersprungen: 03-01-01-01-02-02-01.wav
(5/1440) Uebersprungen: 03-01-02-01-01-01-01.wav
(6/1440) Uebersprungen: 03-01-02-01-01-02-01.wav
(7/1440) Uebersprungen: 03-01-02-01-02-01-01.wav
(8/1440) Uebersprungen: 03-01-02-01-02-02-01.wav
(9/1440) Uebersprungen: 03-01-02-02-01-01-01.wav
(10/1440) Uebersprungen: 03-01-02-02-01-02-01.wav
(11/1440) Uebersprungen: 03-01-02-02-02-01-01.wav
(12/1440) Uebersprungen: 03-01-02-02-02-02-01.wav
(13/1440) Uebersprungen: 03-01-03-01-01-01-01.wav
(14/1440) Uebersprungen: 03-01-03-01-01-02-01.wav
(15/1440) Uebersprungen: 03-01-03-01-02-01-01.wav
(16/1440) Uebersprungen: 03-01-03-01-02-02-01.wav
(17/1440) Uebersprungen: 03-01-03-02-01-01-01.wav
(18/1440) Uebersprungen: 03-01-03-02-01-02-01.wav
(19/1440) Uebersprungen: 03-01-03-02-02-01-01.wav
(20/1440) Uebersprungen: 03-01-03-02-02-02-01.wav
(21/1440)

In [7]:
import json
with open(output_file, "r") as jsonfile:
    data = json.load(jsonfile)
len(data)

1440